# 🤖 Chat with PDF — Backend on Google Colab

This notebook runs the **FastAPI backend** on Colab's servers and exposes it to the internet via **ngrok**, so your local Next.js frontend can call it.

### How it works
```
Your PC (Next.js)  →  ngrok tunnel  →  Colab (FastAPI + Groq)
```

### Steps
1. Run cells in order
2. Copy the ngrok URL printed in **Cell 5**
3. Paste it into `frontend/.env.local` as `NEXT_PUBLIC_API_URL`
4. Start your Next.js frontend locally — done!


## Cell 1 — Install all dependencies

In [ ]:
!pip install -q fastapi uvicorn[standard] python-multipart PyPDF2 \
    sentence-transformers scikit-learn numpy python-dotenv \
    groq pyngrok nest-asyncio requests
print("✅ All packages installed!")

## Cell 2 — Write backend source files to Colab disk

In [ ]:
import os
os.makedirs('/content/backend/data', exist_ok=True)

# ── pdf_processor.py ──────────────────────────────────────────────────────────
pdf_processor_code = '''
import io, PyPDF2
from typing import List

def extract_text_from_pdf(file_bytes: bytes) -> str:
    reader = PyPDF2.PdfReader(io.BytesIO(file_bytes))
    pages = [p.extract_text().strip() for p in reader.pages if p.extract_text()]
    if not pages:
        raise ValueError("No readable text found in the PDF.")
    return "\\n\\n".join(pages)

def split_into_chunks(text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks
'''

# ── embedding_model.py ────────────────────────────────────────────────────────
embedding_model_code = '''
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
_model = None

def _get_model():
    global _model
    if _model is None:
        print(f"[Embedding] Loading {MODEL_NAME}...")
        _model = SentenceTransformer(MODEL_NAME)
        print("[Embedding] Ready!")
    return _model

def get_embedding(text: str) -> np.ndarray:
    return _get_model().encode(text, convert_to_numpy=True)

def get_embeddings(texts: List[str]) -> List[np.ndarray]:
    emb = _get_model().encode(texts, convert_to_numpy=True, show_progress_bar=True)
    return [emb[i] for i in range(len(emb))]
'''

# ── vector_store.py ───────────────────────────────────────────────────────────
vector_store_code = '''
import os, pickle
import numpy as np
from typing import List, Dict, Any
from sklearn.metrics.pairwise import cosine_similarity

DEFAULT_PKL_PATH = "/content/backend/data/my_data.pkl"

def save_store(chunks, embeddings, path=DEFAULT_PKL_PATH):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump({"chunks": chunks, "embeddings": embeddings}, f)
    print(f"[VectorStore] Saved {len(chunks)} chunks to {path}")

def load_store(path=DEFAULT_PKL_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError("No processed PDF found. Upload a PDF first via POST /upload-pdf.")
    with open(path, "rb") as f:
        return pickle.load(f)

def search(query_embedding, store, top_k=5):
    chunks = store["chunks"]
    matrix = np.vstack(store["embeddings"])
    sims = cosine_similarity(query_embedding.reshape(1, -1), matrix)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [chunks[i] for i in top_idx]
'''

# ── chat_engine.py (Groq) ─────────────────────────────────────────────────────
chat_engine_code = '''
import os
from typing import List
from groq import Groq

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
GROQ_MODEL   = os.getenv("GROQ_MODEL", "llama3-8b-8192")

def build_prompt(question: str, chunks: List[str]) -> str:
    context = "\\n\\n---\\n\\n".join(chunks)
    return f"""You are a helpful assistant. Answer ONLY from the document context below.
If the answer is not in the context, say you don\'t know. Be concise.

Context:
{context}

Question: {question}

Answer:"""

def get_answer(question: str, context_chunks: List[str]) -> str:
    if not context_chunks:
        return "No relevant content found in the document for your question."
    if not GROQ_API_KEY:
        raise RuntimeError("GROQ_API_KEY is not set. Set it in Cell 3.")
    client = Groq(api_key=GROQ_API_KEY)
    resp = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": build_prompt(question, context_chunks)}],
        temperature=0.3,
        max_tokens=1024,
    )
    return resp.choices[0].message.content.strip()
'''

# Write all files
files = {
    '/content/backend/pdf_processor.py': pdf_processor_code,
    '/content/backend/embedding_model.py': embedding_model_code,
    '/content/backend/vector_store.py': vector_store_code,
    '/content/backend/chat_engine.py': chat_engine_code,
}
for path, code in files.items():
    with open(path, 'w') as f:
        f.write(code.strip())

print("✅ All backend source files written!")
print(os.listdir('/content/backend'))

## Cell 3 — Set your Groq API Key
Get a free key at https://console.groq.com → API Keys

In [ ]:
import os

# ✏️ PASTE YOUR GROQ API KEY HERE
os.environ['GROQ_API_KEY'] = 'gsk_your_key_here'
os.environ['GROQ_MODEL']   = 'llama3-8b-8192'

print(f"✅ GROQ_API_KEY set: {os.environ['GROQ_API_KEY'][:12]}...")

## Cell 4 — Write and start the FastAPI app

In [ ]:
main_py = '''
import sys
sys.path.insert(0, "/content/backend")

import os
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

from pdf_processor import extract_text_from_pdf, split_into_chunks
from embedding_model import get_embedding, get_embeddings
from vector_store import save_store, load_store, search, DEFAULT_PKL_PATH
from chat_engine import get_answer

app = FastAPI(title="Chat with PDF API (Colab)")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatRequest(BaseModel):
    question: str

@app.get("/health")
def health():
    return {"status": "ok", "provider": "groq", "model": os.getenv("GROQ_MODEL")}

@app.post("/upload-pdf")
async def upload_pdf(file: UploadFile = File(...)):
    if not file.filename.lower().endswith(".pdf"):
        raise HTTPException(400, "Only PDF files accepted.")
    file_bytes = await file.read()
    text   = extract_text_from_pdf(file_bytes)
    chunks = split_into_chunks(text)
    embs   = get_embeddings(chunks)
    save_store(chunks, embs)
    return {"message": f"Processed \'{file.filename}\'", "chunks_count": len(chunks), "filename": file.filename}

@app.post("/chat")
async def chat(req: ChatRequest):
    if not req.question.strip():
        raise HTTPException(400, "Question cannot be empty.")
    try:
        store = load_store()
    except FileNotFoundError as e:
        raise HTTPException(404, str(e))
    q_emb   = get_embedding(req.question)
    chunks  = search(q_emb, store, top_k=5)
    try:
        answer = get_answer(req.question, chunks)
    except RuntimeError as e:
        raise HTTPException(503, str(e))
    return {"answer": answer}
'''

with open('/content/backend/main.py', 'w') as f:
    f.write(main_py.strip())

print("✅ main.py written!")

## Cell 5 — Start FastAPI + ngrok tunnel
⚠️ Set your ngrok authtoken first: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
import nest_asyncio
import uvicorn
import threading
from pyngrok import ngrok, conf

# ✏️ PASTE YOUR NGROK AUTHTOKEN HERE (free at https://dashboard.ngrok.com)
NGROK_AUTH_TOKEN = "your_ngrok_authtoken_here"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

nest_asyncio.apply()

# Start the FastAPI server in a background thread
import sys
sys.path.insert(0, '/content/backend')

def run_server():
    uvicorn.run(
        "main:app",
        host="0.0.0.0",
        port=8000,
        app_dir="/content/backend"
    )

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

import time
time.sleep(3)  # Wait for server to start

# Open ngrok tunnel on port 8000
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print("\n" + "=" * 60)
print(f"🚀 Backend is LIVE at: {public_url}")
print(f"📋 Health check:       {public_url}/health")
print(f"📤 Upload PDF:         {public_url}/upload-pdf")
print(f"💬 Chat:               {public_url}/chat")
print("=" * 60)
print()
print("👉 Copy the URL above and paste it into your frontend/.env.local:")
print(f"   NEXT_PUBLIC_API_URL={public_url}")
print("   Then restart the Next.js dev server: npm run dev")

## Cell 6 — Quick test (optional)
Test the health endpoint to confirm everything is running.

In [ ]:
import requests

# Use the public_url variable from Cell 5
resp = requests.get(f"{public_url}/health")
print("Health check response:", resp.json())
print("✅ Backend is working!" if resp.ok else "❌ Something went wrong")

## Cell 7 — Upload a PDF directly in Colab (optional alternative)
If you want to upload and test a PDF directly from this notebook without using the frontend.

In [ ]:
from google.colab import files as colab_files
import requests

# Upload a file via Colab's file picker
uploaded = colab_files.upload()

for filename, data in uploaded.items():
    print(f"Uploading '{filename}' to the backend...")
    response = requests.post(
        f"{public_url}/upload-pdf",
        files={"file": (filename, data, "application/pdf")}
    )
    print("Response:", response.json())

## Cell 8 — Ask a question (optional test from notebook)

In [ ]:
import requests

question = "What is this document about?"   # ✏️ Change this

resp = requests.post(
    f"{public_url}/chat",
    json={"question": question}
)
data = resp.json()
print(f"Q: {question}")
print(f"A: {data.get('answer', data)}")